# Adult (TabFairGAN) 

Author: Ilse Harmers \
Last modified: February 17, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
from tabfairgan import TFG
import utils
import torch
device = torch.device("cuda:0" if (torch.cuda.is_available() and 1 > 0) else "cpu")
print(device)

In [ ]:
# Importing train set.
adult_train = pd.read_csv("./train-test-datasets/adult/adult_train.csv")
print(adult_train.columns.to_list())
adult_train.describe()

In [ ]:
# Setting up GAN training parameters. 
fairness_config = {
    'fair_epochs': 50,
    'lamda': 0.5,
    'S': 'sex',
    'Y': 'income',
    'S_under': ' Female',
    'Y_desire': ' >50K'
}

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")
    
    # Synthesizing the dataset with TabFairGAN. 
    tfg = TFG(adult_train, epochs = 200, batch_size = 256, device = "cuda:0", fairness_config = fairness_config)
    # Training TabFairGAN synthesizer. 
    tfg.train()

    try:
        # Generating the first synthetic dataset.
        sample1 = tfg.generate_fake_df(adult_train.shape[0])
        # Encoding the sensitive and target attributes for the fairness analysis.
        sample1_encoded = utils.one_hot_encode(sample1[["sex", "income"]], order = [[" <=50K", " >50K"], [" Female", " Male"]])
        dem1 = utils.demographic_parity(df = sample1_encoded, s = "sex", y = "income")
        dis1 = utils.disparate_impact(df = sample1_encoded, s = "sex", y = "income")
        print("Sample 1 [dem., dis.]: ",  [dem1, dis1])
        
        # Generating the second synthetic dataset.
        sample2 = tfg.generate_fake_df(adult_train.shape[0])
        # Encoding the sensitive and target attributes for the fairness analysis.
        sample2_encoded = utils.one_hot_encode(sample2[["sex", "income"]], order = [[" <=50K", " >50K"], [" Female", " Male"]])
        dem2 = utils.demographic_parity(df = sample2_encoded, s = "sex", y = "income")
        dis2 = utils.disparate_impact(df = sample2_encoded, s = "sex", y = "income")
        print("Sample 2 [dem., dis.]: ",  [dem2, dis2])

        # Generating the third synthetic dataset.
        sample3 = tfg.generate_fake_df(adult_train.shape[0])
        # Encoding the sensitive and target attributes for the fairness analysis.
        sample3_encoded = utils.one_hot_encode(sample3[["sex", "income"]], order = [[" <=50K", " >50K"], [" Female", " Male"]])
        dem3 = utils.demographic_parity(df = sample3_encoded, s = "sex", y = "income")
        dis3 = utils.disparate_impact(df = sample3_encoded, s = "sex", y = "income")
        print("Sample 3 [dem., dis.]: ",  [dem3, dis3])

        # Saving the synthetic datasets.
        sample1.to_csv(f"./synthetic-datasets_TabFair/adult/run{r}/sample1.csv", index = False)
        sample2.to_csv(f"./synthetic-datasets_TabFair/adult/run{r}/sample2.csv", index = False)
        sample3.to_csv(f"./synthetic-datasets_TabFair/adult/run{r}/sample3.csv", index = False)
    
        r += 1
    except ZeroDivisionError:
        r += 0